In [1]:
# これはダイクストラ法の基礎理解とコード変換を試すJupiternotebookファイルです。　at 20260424

# Dijkstra法とは？

グラフ理論において、ある1つの始点から他のすべてのノード（頂点）への最短経路（最小コスト）を効率的に求めるアルゴリズム(by wikipedia)

## 計算の基本的な手順：
1.初期化: 始点の距離を0、それ以外の全ノードの距離を無限大に設定。\
2.未確定の探索: 未確定のノードの中で、始点からの距離が最も近いノードuを選択。\
3.距離の更新: 選択したノードvに隣接するノードについて、「（uまでの距離）＋（uからvへのエッジの重み）」が現在のvの距離より小さければ、その距離を更新。\
4.確定: ノードuの最短距離を確定する。
5.繰り返す: 全てのノードが確定するまで2〜4を繰り返す。

## 設計思想について：
貪欲法とDP（動的計画法）の性質も併せもっている\

以下、geminiの解説
【貪欲法の性質】
先ほど解説した通り、**「未確定の中で一番コストが小さいノードを、迷わず確定させる」**という部分は完全に貪欲法です。これがダイクストラ法の「一歩一歩の進み方」を定義しています。

【DP（動的計画法）の性質】
DPの本質は**「部分問題の解を再利用する」**ことにあります。
ダイクストラ法も同じことをしています。

あるノード B までの最短コストを計算する際、「すでに計算済みのノード A までの最短コスト」という過去の結果を利用して、A → B の距離を足し算しています。
ダイクストラ法というアルゴリズムは、「辺が負にならない」という特殊な条件があるおかげで、たまたま「貪欲法的な突き進み方」をしても、結果的に「DPで全探索したのと同じ（正確な）結果」が得られるという、奇跡的なバランスの上に成り立っているアルゴリズムなのです。

「一度確定した最短コストは二度と計算し直さない」という仕組みは、DPの「メモ化」と全く同じ考え方です。

## 自身の中で覚えておけばいいこと
・確定距離がある点と隣り合っている点の暫定距離を出す→最短の暫定距離を確定距離とする。を繰り返す。\
・オーダー（計算量）はO(n!)と点の数に応じて階乗で上がっていく。→規模に注意して使う。\
・最適解を保証している。\
・実装自体は簡単（らしい）\
・貪欲法とDP（動的計画法）の性質も併せもっている\
・点ごとには貪欲法。今までの最適をもとにしているからDPとも言える。

# コード化　ムズイ（正味ライブラリ流用でいいと考えます）

In [4]:
# 正解コード
def dijkstra(edges, num_node):
    """ 経路の表現
            [終点, 辺の値]
            A, B, C, D, ... → 0, 1, 2, ...とする """
    node = [float('inf')] * num_node    #スタート地点以外の値は∞で初期化
    node[0] = 0     #スタートは0で初期化

    node_name = [i for i in range(num_node)]     #ノードの名前を0~ノードの数で表す

    while len(node_name) > 0:
        r = node_name[0]

        #最もコストが小さい頂点を探す
        for i in node_name:
            if  node[i] < node[r]:
                r = i   #コストが小さい頂点が見つかると更新

        #最もコストが小さい頂点を取り出す
        min_point = node_name.pop(node_name.index(r))

        #経路の要素を各変数に格納することで，視覚的に見やすくする
        for factor in edges[min_point]:
            goal = factor[0]   #終点
            cost  = factor[1]   #コスト

            #更新条件
            if node[min_point] + cost < node[goal]:
                node[goal] = node[min_point] + cost     #更新

    return node

if __name__ == '__main__':
    Edges = [
        [[1, 4], [2, 3]],             # ← 頂点Aからの辺のリスト
        [[2, 1], [3, 1], [4, 5]],   # ← 頂点Bからの辺のリスト
        [[5, 2]],                       # ← 頂点Cからの辺のリスト
        [[4, 3]],                       # ← 頂点Dからの辺のリスト
        [[6, 2]],                       # ← 頂点Eからの辺のリスト
        [[4, 1], [6, 4]],             # ← 頂点Fからの辺のリスト
        []                                # ← 頂点Gからの辺のリスト
        ]
    
    #今の目的地の数は7つ（0~6: A~G）
    node_num = 7

    opt_node = dijkstra(Edges, node_num)

    #以下は結果を整理するためのコード
    node_name = []
    for i in range(node_num):
        node_name.append(chr(ord('A') + i))    
    result = []
    for i in range(len(opt_node)):
        result.append(f"{node_name[i]} : {opt_node[i]}")
    print(f"'目的地:そこまでの最小コスト'\n\n{result}")

# これはリストであらわしてる


'目的地:そこまでの最小コスト'

['A : 0', 'B : 4', 'C : 3', 'D : 5', 'E : 6', 'F : 5', 'G : 8']


In [5]:
# 経路記録
"""
2021/01/29
@Yuya Shimizu

ダイクストラ法（ヒープによる優先度付きキューを用いて）
経路を記録する
"""
import heapq

def dijkstra(edges, num_node, Goal):
    """ 経路の表現
            [終点, 辺の値]
            A, B, C, D, ... → 0, 1, 2, ...とする """
    node = [float('inf')] * num_node    #スタート地点以外の値は∞で初期化
    node[0] = 0     #スタートは0で初期化

    node_name = []
    heapq.heappush(node_name, [0, [0]])

    while len(node_name) > 0:
        #ヒープから取り出し
        _, min_point = heapq.heappop(node_name)
        last = min_point[-1]
        if last == Goal:
            return min_point, node  #道順とコストを出力させている
        
        #経路の要素を各変数に格納することで，視覚的に見やすくする
        for factor in edges[last]:
            goal = factor[0]   #終点
            cost  = factor[1]   #コスト

            #更新条件
            if node[last] + cost < node[goal]:
                node[goal] = node[last] + cost     #更新
                #ヒープに登録
                heapq.heappush(node_name, [node[last] + cost, min_point + [goal]])

    return []

if __name__ == '__main__':
    Edges = [
        [[1, 4], [2, 3]],             # ← 頂点Aからの辺のリスト
        [[2, 1], [3, 1], [4, 5]],   # ← 頂点Bからの辺のリスト
        [[5, 2]],                       # ← 頂点Cからの辺のリスト
        [[4, 3]],                       # ← 頂点Dからの辺のリスト
        [[6, 2]],                       # ← 頂点Eからの辺のリスト
        [[4, 1], [6, 4]],             # ← 頂点Fからの辺のリスト
        []                                # ← 頂点Gからの辺のリスト
        ]
    
    #今の目的地の数は7つ（0~6: A~G）
    node_num = 7
    Goal = 6    
    opt_root, opt_cost = dijkstra(Edges, node_num, Goal)    #道順とコストを出力させている

    #出力を見やすく整理するための変換用辞書型リストの作成
    root_converter = {}
    cost_converter = {}
    for i in range(node_num):
        root_converter[i] = chr(ord('A') + i)
        cost_converter[i] = opt_cost[i]
    
    arrow = " → "
    result = ""
    for i in range(len(opt_root)):
        if i > 0:
            result += arrow
        result += f"{root_converter[opt_root[i]]}({cost_converter[opt_root[i]]})"
        
    print(f"ノード(そこまでのコスト)\n\n{result}")


ノード(そこまでのコスト)

A(0) → C(3) → F(5) → E(6) → G(8)
